In [11]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
from urllib.parse import urljoin

# Load environment variables and set up Groq API
load_dotenv()
openai = OpenAI(
    api_key=os.getenv('GROQ_API_KEY'),
    base_url="https://api.groq.com/openai/v1"
)

# Week 1 Practice: AI-Powered Product Review Comparison

This notebook demonstrates how to use **web scraping + AI** to find the best-quality product based on reviews.

**Workflow:**
1. **Input**: Product name (e.g., "wireless headphones")
2. **Web Scraping**: Search multiple popular product sites
3. **Review Collection**: Extract review information and ratings
4. **AI Analysis**: Use LLM to analyze reviews and pick the best quality product
5. **Output**: Best product recommendation with quality analysis

**Key Concepts:**
- Web scraping from e-commerce sites
- Review data extraction
- AI-powered product comparison
- JSON-structured responses
- Real-world e-commerce automation

Let's build a smart product finder!

In [12]:
# Define popular product search sites
PRODUCT_SITES = [
    "https://www.amazon.com",
    "https://www.bestbuy.com",
    "https://www.walmart.com",
    "https://www.target.com",
    "https://www.newegg.com",
    "https://www.bhphotovideo.com",
    "https://www.adorama.com",
    "https://www.costco.com",
    "https://www.macys.com",
    "https://www.ebay.com"
]

def search_product_on_sites(product_name):
    """
    Search for a product on multiple popular e-commerce sites.
    Returns a list of search results with product links.
    """
    search_results = []
    
    for site in PRODUCT_SITES:
        # Build search URL based on site
        if "amazon" in site:
            search_url = f"{site}/s?k={product_name.replace(' ', '+')}"
        elif "bestbuy" in site:
            search_url = f"{site}/site/searchpage.jsp?st={product_name.replace(' ', '+')}"
        elif "walmart" in site:
            search_url = f"{site}/search?q={product_name.replace(' ', '+')}"
        elif "target" in site:
            search_url = f"{site}/s?searchTerm={product_name.replace(' ', '+')}"
        elif "ebay" in site:
            search_url = f"{site}/sch/i.html?_nkw={product_name.replace(' ', '+')}"
        else:
            search_url = f"{site}/search?q={product_name.replace(' ', '+')}"
        
        try:
            content = fetch_website_contents(search_url)
            if content:
                search_results.append({
                    "site": site,
                    "search_url": search_url,
                    "content_preview": content[:500]  # Preview of page content
                })
        except:
            pass
    
    return search_results

In [13]:
product_analysis_prompt = """
You are an expert product analyst. You have collected information about a product from multiple e-commerce websites, including reviews and ratings.

Your task is to:
1. Analyze the quality of each product based on reviews
2. Identify the product with the BEST QUALITY (not price)
3. Provide a JSON response with the recommended product

Focus on:
- Customer satisfaction and ratings
- Product durability and reliability
- Feature quality and functionality
- Review authenticity and detailed feedback

Ignore price comparisons - we only care about quality.

Respond in JSON format like this:
{
    "best_product": {
        "name": "Product Name",
        "site": "website name",
        "quality_score": 9.5,
        "reasons": ["reason 1", "reason 2", "reason 3"],
        "review_highlights": "Summary of what customers loved"
    },
    "analysis": "Brief explanation of why this product won"
}
"""

In [14]:
def build_product_analysis_prompt(product_name, search_results):
    """
    Build a user prompt with product information from multiple sites.
    """
    prompt = f"""
Product to analyze: {product_name}

I have collected product information from multiple e-commerce websites. 
Analyze each product based on reviews and quality metrics, then recommend the BEST QUALITY product.

Here's the data collected from {len(search_results)} sites:

"""
    
    for i, result in enumerate(search_results, 1):
        prompt += f"\n--- Site {i}: {result['site']} ---\n"
        prompt += result['content_preview'][:1000]  # Include preview of website content
        prompt += "\n"
    
    prompt += "\nBased on this information, identify the product with the BEST QUALITY (regardless of price)."
    prompt += "\nRespond in the specified JSON format."
    
    return prompt[:5000]  # Limit to avoid token issues


def find_best_quality_product(product_name):
    """
    Search for a product, collect reviews, and use AI to find the best quality product.
    """
    print(f"Searching for '{product_name}' on major e-commerce sites...")
    
    # Search on multiple sites
    search_results = search_product_on_sites(product_name)
    print(f"Found results from {len(search_results)} sites")
    
    if len(search_results) == 0:
        print("No results found.")
        return None
    
    # Build analysis prompt
    user_prompt = build_product_analysis_prompt(product_name, search_results)
    
    # Call AI to analyze and pick best product
    print("Analyzing products with AI...")
    response = openai.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": product_analysis_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format={"type": "json_object"}
    )
    
    result = response.choices[0].message.content
    product_recommendation = json.loads(result)
    
    return product_recommendation

## Step 1: Search for Products

The `find_best_quality_product()` function:
1. **Searches** the product on 10 major e-commerce sites
2. **Collects** reviews and product information
3. **Uses AI** to analyze reviews and pick the best quality
4. **Returns** a JSON recommendation with reasoning

Let's try it with a product!

In [18]:
# Example: Find the best wireless headphones
product_name = "wireless headphones"
recommendation = find_best_quality_product(product_name)

if recommendation:
    best = recommendation.get("best_product", {})
    print("\n" + "="*60)
    print(f"🏆 BEST QUALITY {product_name.upper()}")
    print("="*60)
    print(f"Product: {best.get('name', 'N/A')}")
    print(f"Site: {best.get('site', 'N/A')}")
    print(f"Quality Score: {best.get('quality_score', 'N/A')}/10")
    print(f"\nWhy this product:")
    for reason in best.get('reasons', []):
        print(f"  ✓ {reason}")
    print(f"\nCustomer Review Highlights:")
    print(f"  {best.get('review_highlights', 'N/A')}")
    print(f"\nAnalysis:")
    print(f"  {recommendation.get('analysis', 'N/A')}")
    print("="*60)

Searching for 'wireless headphones' on major e-commerce sites...
Error fetching content from https://www.amazon.com/s?k=wireless+headphones: 503 Server Error: Service Unavailable for url: https://www.amazon.com/s?k=wireless+headphones
Error fetching content from https://www.bestbuy.com/site/searchpage.jsp?st=wireless+headphones: 403 Client Error: Forbidden for url: https://www.bestbuy.com/site/searchpage.jsp?st=wireless+headphones
Error fetching content from https://www.bhphotovideo.com/search?q=wireless+headphones: 403 Client Error: Forbidden for url: https://www.bhphotovideo.com/search?q=wireless+headphones
Error fetching content from https://www.adorama.com/search?q=wireless+headphones: 403 Client Error: Forbidden for url: https://www.adorama.com/search?q=wireless+headphones
Error fetching content from https://www.costco.com/search?q=wireless+headphones: 403 Client Error: Forbidden for url: https://www.costco.com/search?q=wireless+headphones
Error fetching content from https://www.m

## Step 2: Try Different Products

You can search for any product by changing the `product_name` variable below and running the cell.

**Examples to try:**
- `"laptop"`
- `"smartphone"`
- `"gaming monitor"`
- `"mechanical keyboard"`
- `"running shoes"`

Just replace the product name and run the cell!

In [16]:
# Try a different product
# Just change the product_name and run this cell

product_name = "blender"  # Change this to any product you want to analyze

In [17]:

recommendation = find_best_quality_product(product_name)

if recommendation:
    best = recommendation.get("best_product", {})
    print("\n" + "="*60)
    print(f"🏆 BEST QUALITY {product_name.upper()}")
    print("="*60)
    print(f"Product: {best.get('name', 'N/A')}")
    print(f"Site: {best.get('site', 'N/A')}")
    print(f"Quality Score: {best.get('quality_score', 'N/A')}/10")
    print(f"\nWhy this product:")
    for reason in best.get('reasons', []):
        print(f"  ✓ {reason}")
    print(f"\nCustomer Review Highlights:")
    print(f"  {best.get('review_highlights', 'N/A')}")
    print(f"\nAnalysis:")
    print(f"  {recommendation.get('analysis', 'N/A')}")
    print("="*60)

Searching for 'blender' on major e-commerce sites...
Error fetching content from https://www.amazon.com/s?k=blender: 503 Server Error: Service Unavailable for url: https://www.amazon.com/s?k=blender
Error fetching content from https://www.bestbuy.com/site/searchpage.jsp?st=blender: 403 Client Error: Forbidden for url: https://www.bestbuy.com/site/searchpage.jsp?st=blender
Error fetching content from https://www.bhphotovideo.com/search?q=blender: 403 Client Error: Forbidden for url: https://www.bhphotovideo.com/search?q=blender
Error fetching content from https://www.adorama.com/search?q=blender: 403 Client Error: Forbidden for url: https://www.adorama.com/search?q=blender
Error fetching content from https://www.costco.com/search?q=blender: 403 Client Error: Forbidden for url: https://www.costco.com/search?q=blender
Error fetching content from https://www.macys.com/search?q=blender: 403 Client Error: Forbidden for url: https://www.macys.com/search?q=blender
Found results from 4 sites
An

## Key Takeaways

✅ **Web Scraping Pipeline** - Extract product data from multiple e-commerce sites  
✅ **AI-Powered Analysis** - Use LLM to understand reviews and quality metrics  
✅ **Structured Output** - JSON responses make results machine-readable  
✅ **Quality-First** - AI ignores price and focuses on customer satisfaction & product quality  
✅ **Real-World Application** - Automated product comparison for smarter shopping  

### How It Works:

1. **Search**: Scrape product pages from 10+ e-commerce sites
2. **Analyze**: Extract reviews, ratings, and customer feedback
3. **Compare**: Use AI to evaluate quality across all products
4. **Recommend**: AI returns the best quality product with reasoning
5. **Output**: Structured JSON with quality score and analysis

### Use Cases:
- Smart shopping assistant
- Product quality comparison
- E-commerce recommendation engine
- Automated review analysis
- Quality-focused product finder

